# 02 - Protótipo da Base de Treino

Objetivo deste notebook: testar a preparação da base que será usada em um treino futuro, sem treinar o modelo nesta etapa.

A entrada será o banco SQLite criado a partir da análise anterior. O foco aqui é validar limpeza, tratamento de nulos, codificação de categorias e normalização das variáveis numéricas.

## 1. Configuração inicial

Vamos carregar as bibliotecas necessárias e definir o caminho do banco local. Como a tabela é grande, este notebook usa uma amostra de usuários para testar a ideia com segurança.

In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# Caminhos e parâmetros principais do protótipo.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATABASE_PATH = PROJECT_ROOT / "data" / "training_data.db"
TABLE_NAME = "training_data"

SAMPLE_USER_LIMIT = 5000
RANDOM_STATE = 42

DATABASE_PATH

## 2. Leitura da base

A tabela `training_data` já representa pares `user_id` e `product_id`, com variáveis agregadas e a coluna `target`. Primeiro vamos conferir o volume total e a distribuição do alvo no banco.

In [ ]:
# Consulta o volume total e a distribuição do alvo sem carregar a tabela inteira.
with sqlite3.connect(DATABASE_PATH) as conn:
    total_rows = pd.read_sql_query(f"SELECT COUNT(*) AS rows FROM {TABLE_NAME}", conn)
    target_distribution = pd.read_sql_query(
        f"""
        SELECT target, COUNT(*) AS rows
        FROM {TABLE_NAME}
        GROUP BY target
        ORDER BY target
        """,
        conn,
    )

target_distribution["rate"] = target_distribution["rows"] / target_distribution["rows"].sum()

display(total_rows)
display(target_distribution)

Agora carregamos uma amostra por usuários. Isso preserva várias linhas do mesmo cliente e mantém a estrutura natural da recomendação, em vez de sortear linhas totalmente soltas.

In [ ]:
# Carrega uma amostra por usuário, preservando vários pares usuário-produto do mesmo cliente.
sample_query = f"""
SELECT *
FROM {TABLE_NAME}
WHERE user_id IN (
    SELECT DISTINCT user_id
    FROM {TABLE_NAME}
    LIMIT {SAMPLE_USER_LIMIT}
)
"""

with sqlite3.connect(DATABASE_PATH) as conn:
    data = pd.read_sql_query(sample_query, conn)

data.head()

## 3. Validações iniciais

Antes de aplicar transformações, vamos verificar tamanho da amostra, tipos de dados, nulos, duplicados e distribuição do alvo.

In [ ]:
# Visão rápida do tamanho da amostra.
summary = pd.DataFrame([
    {
        "rows": len(data),
        "columns": data.shape[1],
        "memory_mb": data.memory_usage(deep=True).sum() / 1024**2,
        "users": data["user_id"].nunique(),
        "products": data["product_id"].nunique(),
    }
])

summary

In [ ]:
# Tipos, nulos e cardinalidade por coluna.
quality_report = pd.DataFrame(
    {
        "dtype": data.dtypes.astype(str),
        "nulls": data.isna().sum(),
        "null_rate": data.isna().mean(),
        "unique_values": data.nunique(dropna=True),
    }
).reset_index(names="column")

quality_report

In [ ]:
# Checagens simples antes da preparação final.
validation_report = pd.DataFrame([
    {"check": "duplicated_user_product", "rows": data.duplicated(["user_id", "product_id"]).sum()},
    {"check": "invalid_target", "rows": (~data["target"].isin([0, 1])).sum()},
    {"check": "missing_user_id", "rows": data["user_id"].isna().sum()},
    {"check": "missing_product_id", "rows": data["product_id"].isna().sum()},
])

target_rate = data["target"].value_counts(normalize=True).rename("rate").to_frame()

display(validation_report)
display(target_rate)

## 4. Separação das colunas

`user_id` e `product_id` são identificadores. Eles serão preservados para rastreabilidade, mas não serão normalizados como se fossem variáveis numéricas comuns.

`aisle_id` e `department_id` também não entram como features, porque são códigos de categoria e duplicam a informação textual de `aisle` e `department`. Para este protótipo, usamos apenas os nomes das categorias no encoder.

In [ ]:
# IDs são guardados para rastreabilidade, mas não entram no preparo das features.
identifier_columns = ["user_id", "product_id"]
target_column = "target"

# `aisle` e `department` serão transformados em colunas binárias.
categorical_columns = ["aisle", "department"]

# Variáveis numéricas já agregadas no notebook anterior.
numeric_columns = [
    "purchase_count",
    "reorder_rate",
    "avg_cart_position",
    "last_order_number",
    "user_total_orders",
    "user_unique_products",
    "user_reorder_rate",
    "user_total_purchases",
    "product_unique_users",
    "product_total_orders",
    "product_reorder_rate",
]

X = data[numeric_columns + categorical_columns]
y = data[target_column]
ids = data[identifier_columns]

X.head()

## 5. Limpeza e normalização com scikit-learn

A preparação final usa `Pipeline` e `ColumnTransformer`. Assim, o mesmo processo poderá ser reaproveitado no treino futuro, aplicando `fit` apenas nos dados de treino e `transform` nos demais conjuntos.

Neste protótipo, as variáveis numéricas serão normalizadas com `MinMaxScaler`, ficando entre 0 e 1. As variáveis `aisle` e `department` serão convertidas em colunas binárias com `OneHotEncoder`.

In [ ]:
# Divide a amostra mantendo a proporção do target.
X_train, X_valid, y_train, y_valid, ids_train, ids_valid = train_test_split(
    X,
    y,
    ids,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_valid.shape

In [ ]:
# Numéricas: preenche nulos com mediana e normaliza para o intervalo 0-1.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
])

# Categóricas: preenche nulos e cria colunas binárias.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns),
])

# Faz o scikit-learn devolver DataFrames, facilitando a leitura do resultado.
preprocessor.set_output(transform="pandas")

O `fit` é feito apenas na parte de treino da amostra. Isso evita vazamento de informação e simula o fluxo correto que será usado quando houver treinamento de modelo.

In [ ]:
# Aprende as regras no treino e aplica as mesmas regras na validação.
X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)

X_train_prepared.head()

## 6. Checagem da base preparada

Depois da transformação, vamos confirmar se a base não possui nulos, infinitos ou inconsistências de formato. Também vamos validar se as colunas criadas a partir de `aisle` e `department` são realmente binárias.

In [ ]:
# Confirma se a matriz final está limpa e dentro da escala esperada.
prepared_report = pd.DataFrame([
    {
        "dataset": "train",
        "rows": len(X_train_prepared),
        "columns": X_train_prepared.shape[1],
        "nulls": X_train_prepared.isna().sum().sum(),
        "infinite_values": np.isinf(X_train_prepared.to_numpy()).sum(),
        "min_value": X_train_prepared.min().min(),
        "max_value": X_train_prepared.max().max(),
        "target_rate": y_train.mean(),
    },
    {
        "dataset": "valid",
        "rows": len(X_valid_prepared),
        "columns": X_valid_prepared.shape[1],
        "nulls": X_valid_prepared.isna().sum().sum(),
        "infinite_values": np.isinf(X_valid_prepared.to_numpy()).sum(),
        "min_value": X_valid_prepared.min().min(),
        "max_value": X_valid_prepared.max().max(),
        "target_rate": y_valid.mean(),
    },
])

prepared_report

In [ ]:
# As colunas de `aisle` e `department` devem ser binárias após o OneHotEncoder.
encoded_columns = X_train_prepared.filter(regex="^cat__").columns

binary_encoding_report = pd.DataFrame([
    {
        "encoded_categorical_columns": len(encoded_columns),
        "only_binary_values": X_train_prepared[encoded_columns].isin([0, 1]).all().all(),
    }
])

binary_encoding_report

In [ ]:
# Base de protótipo pronta para inspeção ou treino futuro.
train_dataset_prototype = pd.concat(
    [
        ids_train.reset_index(drop=True),
        X_train_prepared.reset_index(drop=True),
        y_train.reset_index(drop=True).rename("target"),
    ],
    axis=1,
)

train_dataset_prototype.head()

## 7. Conclusão

Este notebook valida a ideia de preparação da base para treino futuro. A base parte do SQLite já materializado, separa identificadores e alvo, aplica tratamento de nulos, normalização numérica e codificação categórica com `scikit-learn`.

Neste protótipo, `aisle_id` e `department_id` não são usados como features porque duplicam `aisle` e `department`. As variáveis numéricas ficam normalizadas entre 0 e 1, enquanto `aisle` e `department` viram colunas binárias. O próximo passo será decidir quais variáveis ficam na versão final, avaliar o tamanho da base completa e então criar o notebook ou pipeline de treinamento.